# Phase 2.4 — Report & consolidate (5 models x 3 seeds)

Turns the 15 per-sentence CSVs from Phase 2.3 into the deliverables, and acts as the
**validation gate before Phase 3**. It does not retrain anything — it reads `preds/*_seed*.csv`.

Produces:
- **Integrity checks** — every file covers the *same* test `sentence_id` set, `true_label` agrees
  across models per id, no NaNs. (If these fail, Phase 4's joins are unsafe — fix before proceeding.)
- **`results/sa_per_run_metrics.csv`** — accuracy, macro-F1, weighted-F1, per-class F1 for each run.
- **`results/sa_results_table.csv`** + a paper-formatted `..._formatted.csv` — **mean ± std over seeds**.
- **`figures/sa_macro_f1_by_model.png`** — macro-F1 by model with std error bars.
- **`preds/all_predictions_long.csv`** — all runs stacked into one tidy table for Phase 4.

Works on **partial results too** — if the sweep isn't finished it reports what's present and flags the gaps.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score

PROJECT = "/content/drive/MyDrive/codemix_project"
DATA    = f"{PROJECT}/data"
PREDS   = f"{PROJECT}/preds"
RESULTS = f"{PROJECT}/results"
FIGURES = f"{PROJECT}/figures"
for d in (RESULTS, FIGURES):
    os.makedirs(d, exist_ok=True)

EXPECTED_MODELS = ["mbert", "muril", "indic-bert", "hing-bert", "hing-roberta"]
EXPECTED_SEEDS  = [42, 43, 44]
HINGLISH        = {"hing-bert", "hing-roberta"}

with open(f"{DATA}/label_mapping.json") as f:
    label2id = json.load(f)
id2label    = {int(v): k for k, v in label2id.items()}
CLASS_NAMES = [id2label[i] for i in range(len(id2label))]     # e.g. ['negative','neutral','positive']
print("classes:", CLASS_NAMES)

In [ ]:
# --- load every per-run CSV (skip the consolidated file if it already exists) --
paths = sorted(p for p in glob.glob(f"{PREDS}/*_seed*.csv"))
assert paths, f"no prediction CSVs in {PREDS} - run the Phase 2.3 sweep first"

allp = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
req = {"sentence_id", "model", "seed", "true_label", "pred_label", "correct"}
assert req.issubset(allp.columns), f"missing columns: {req - set(allp.columns)}"
print(f"loaded {len(paths)} CSVs -> {len(allp)} prediction rows")

# presence matrix over the expected grid
present = pd.DataFrame(False, index=EXPECTED_MODELS, columns=EXPECTED_SEEDS)
for (m, s), _ in allp.groupby(["model", "seed"]):
    if m in present.index and s in present.columns:
        present.loc[m, s] = True
print("\nrun presence:")
print(present.replace({True: "OK", False: "--"}).to_string())
missing = [(m, s) for m in EXPECTED_MODELS for s in EXPECTED_SEEDS if not present.loc[m, s]]
print("\nall 15 present" if not missing else f"\nMISSING (report is partial): {missing}")

In [ ]:
# --- integrity gate: the things Phase 4 relies on ----------------------------
id_sets = {os.path.basename(p): set(pd.read_csv(p)["sentence_id"]) for p in paths}
ref_name, ref_ids = next(iter(id_sets.items()))
diffs = {n: len(ids ^ ref_ids) for n, ids in id_sets.items() if ids != ref_ids}
if diffs:
    print("!! sentence_id sets differ across files (would break Phase 4 joins):")
    for n, k in diffs.items():
        print(f"   {n}: {k} differing ids vs {ref_name}")
else:
    print(f"OK: all files share the same {len(ref_ids)} test sentence_ids")

tl = allp.groupby("sentence_id")["true_label"].nunique()
assert (tl == 1).all(), "true_label disagrees across files for some sentence_id"
assert allp[["true_label", "pred_label"]].notna().all().all(), "NaN labels present"
# recompute 'correct' from scratch as a guard against a stale column
assert (allp["correct"] == (allp.true_label == allp.pred_label).astype(int)).all(), "'correct' column stale"
print("OK: true_label consistent, no NaNs, 'correct' column verified")

In [ ]:
# --- per-run metrics ---------------------------------------------------------
labels = list(range(len(CLASS_NAMES)))
rows = []
for (m, s), g in allp.groupby(["model", "seed"]):
    y, yhat = g.true_label, g.pred_label
    per_class = f1_score(y, yhat, average=None, labels=labels)
    row = {"model": m, "seed": s,
           "accuracy":    accuracy_score(y, yhat),
           "f1_macro":    f1_score(y, yhat, average="macro"),
           "f1_weighted": f1_score(y, yhat, average="weighted")}
    for name, val in zip(CLASS_NAMES, per_class):
        row[f"f1_{name}"] = val
    rows.append(row)

per_run = pd.DataFrame(rows).sort_values(["model", "seed"])
per_run.to_csv(f"{RESULTS}/sa_per_run_metrics.csv", index=False)
print(per_run.round(4).to_string(index=False))

In [ ]:
# --- aggregate: mean +/- std over seeds --------------------------------------
metric_cols = ["accuracy", "f1_macro", "f1_weighted"] + [f"f1_{c}" for c in CLASS_NAMES]
agg = per_run.groupby("model")[metric_cols].agg(["mean", "std"])
agg.columns = [f"{a}_{b}" for a, b in agg.columns]
agg = (agg.reindex(EXPECTED_MODELS).dropna(how="all")
          .sort_values("f1_macro_mean", ascending=False).reset_index())
agg.round(4).to_csv(f"{RESULTS}/sa_results_table.csv", index=False)

def pm(mean, std):
    return f"{mean*100:.1f} \u00b1 {std*100:.1f}" if pd.notna(std) else f"{mean*100:.1f}"

paper = pd.DataFrame({"model": agg["model"]})
paper["Accuracy"] = [pm(m, s) for m, s in zip(agg["accuracy_mean"], agg["accuracy_std"])]
paper["Macro-F1"] = [pm(m, s) for m, s in zip(agg["f1_macro_mean"], agg["f1_macro_std"])]
for c in CLASS_NAMES:
    paper[f"F1[{c}]"] = [pm(m, s) for m, s in zip(agg[f"f1_{c}_mean"], agg[f"f1_{c}_std"])]
paper.to_csv(f"{RESULTS}/sa_results_table_formatted.csv", index=False)

print("=== main results (macro-F1 mean +/- std over seeds) ===")
print(paper.to_string(index=False))

In [ ]:
# --- figure: macro-F1 by model with std error bars ---------------------------
order = agg["model"].tolist()
means = (agg["f1_macro_mean"] * 100).values
stds  = (agg["f1_macro_std"].fillna(0) * 100).values
colors = ["#2b6cb0" if m in HINGLISH else "#a0aec0" for m in order]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(order, means, yerr=stds, capsize=4, color=colors)
ax.set_ylabel("Macro-F1 (%)")
ax.set_title("SentiMix SA \u2014 macro-F1 by model (mean \u00b1 std, 3 seeds)")
ax.set_ylim(0, means.max() + 10)
for i, (mu, sd) in enumerate(zip(means, stds)):
    ax.text(i, mu + sd + 0.6, f"{mu:.1f}", ha="center", fontsize=9)
handles = [plt.Rectangle((0,0),1,1,color="#2b6cb0"), plt.Rectangle((0,0),1,1,color="#a0aec0")]
ax.legend(handles, ["Hinglish-specific", "general"], loc="lower left", fontsize=8)
plt.xticks(rotation=20); plt.tight_layout()
fig.savefig(f"{FIGURES}/sa_macro_f1_by_model.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved -> {FIGURES}/sa_macro_f1_by_model.png")

In [ ]:
# --- consolidate all runs into one tidy table for Phase 4 --------------------
# (named without '_seed' so the loader glob above never re-ingests it)
allp_long = allp.sort_values(["model", "seed", "sentence_id"]).reset_index(drop=True)
out = f"{PREDS}/all_predictions_long.csv"
allp_long.to_csv(out, index=False)
print(f"consolidated {len(allp_long)} rows ({allp_long.model.nunique()} models x "
      f"{allp_long.seed.nunique()} seeds x {allp_long.sentence_id.nunique()} sentences) -> {out}")
allp_long.head()

## Read this before Phase 3

- **Integrity gate must be clean** — matching `sentence_id` sets, consistent `true_label`, no NaNs.
  Everything downstream joins on `sentence_id`; a red flag here is a stop-and-fix, not a warning to skip.
- **Sanity of the ranking:** Hinglish models on top, IndicBERT lower (expected, not a bug), nothing at ~33 F1.
- Keep an eye on **per-class F1** — a decent macro-F1 can still hide one weak class (often `neutral`);
  Phase 5's qualitative pass will want to know which.

## Next — Phase 3 (feature extraction)

Build `features/sentence_features.csv` keyed by `sentence_id`: cmi (already in the master),
switch_density (from `lang_tags`), length, hi_ratio, oov_rate, caps_ratio. Then merge onto
`all_predictions_long.csv` on `sentence_id` and assert row counts — that merged frame is what
Phase 4's point-biserial, accuracy-by-CMI-bucket, and the robustness-gap analysis run on.
